In [48]:
# ------------------------------------------------------------
# 경고 제거
# ------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ============================================================
# 전자상거래 성과 분석 대시보드
# Google Colab 최종버전
# ============================================================

!apt-get -qq update
!apt-get -qq install fonts-nanum
!pip -q install openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from google.colab import files

# ============================================================
# 한글 폰트 설정
# ============================================================

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [75]:
# ============================================================
# 고객 이탈 위험도 분석
# ============================================================

import pandas as pd
from google.colab import files
from IPython.display import display

# ============================================================
# 파일 업로드
# ============================================================

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

# ============================================================
# 날짜 처리
# ============================================================

df['날짜'] = pd.to_datetime(df['날짜'])
df['연월'] = df['날짜'].dt.to_period('M')

# ============================================================
# 월별 집계
# ============================================================

monthly = (
    df.groupby(['고객 ID','연월'])
      .agg(
          접속여부=('접속 여부','max'),
          기능사용=('핵심 기능 이용 횟수','sum')
      )
      .reset_index()
)

# ============================================================
# 고객별 계산
# ============================================================

results = []

for customer in sorted(monthly['고객 ID'].unique()):

    temp = (
        monthly[monthly['고객 ID']==customer]
        .sort_values('연월')
        .reset_index(drop=True)
    )

    if len(temp) < 2:
        continue

    prev = temp.iloc[-2]
    curr = temp.iloc[-1]

    prev_login = int(prev['접속여부'])
    curr_login = int(curr['접속여부'])

    prev_feature = int(prev['기능사용'])
    curr_feature = int(curr['기능사용'])

    # --------------------------------------------------------
    # 접속 점수
    # --------------------------------------------------------

    login_score = 40 if curr_login == 0 else 0

    # --------------------------------------------------------
    # 기능 점수
    # --------------------------------------------------------

    if prev_feature > 0:

        feature_score = (
            (prev_feature - curr_feature)
            / prev_feature
        ) * 60

        feature_score = max(0, feature_score)

    else:
        feature_score = 0

    # --------------------------------------------------------
    # 최종 점수
    # --------------------------------------------------------

    churn_score = round(
        login_score + feature_score,
        1
    )

    churn_score = min(100, churn_score)

    # --------------------------------------------------------
    # 위험 단계
    # --------------------------------------------------------

    if churn_score < 30:
        stage = "충성"

    elif churn_score < 50:
        stage = "일반"

    elif churn_score < 80:
        stage = "위험"

    else:
        stage = "이탈"

    # --------------------------------------------------------
    # 분석평
    # --------------------------------------------------------

    if stage == "충성":
        comment = "활동이 안정적으로 유지되고 있음"

    elif stage == "일반":
        comment = "일부 활동 감소가 관찰됨"

    elif stage == "위험":
        comment = "핵심 기능 사용 감소가 뚜렷함"

    else:
        comment = "접속 및 기능 사용이 사실상 중단됨"

    results.append([
        customer,
        prev_login,
        curr_login,
        prev_feature,
        curr_feature,
        churn_score,
        stage,
        comment
    ])

# ============================================================
# 결과 생성
# ============================================================

result_df = pd.DataFrame(
    results,
    columns=[
        '고객 ID',
        '지난달 접속 여부',
        '이번달 접속 여부',
        '지난달 기능 사용',
        '이번달 기능 사용',
        'Churn Score',
        '이탈 위험 단계',
        '분석 평'
    ]
)

# ============================================================
# 출력
# ============================================================

print("\n고객 이탈 위험도 분석 결과\n")

display(result_df)

# ============================================================
# 저장
# ============================================================

output_file = "고객_이탈위험도_분석결과.xlsx"

result_df.to_excel(
    output_file,
    index=False
)

files.download(output_file)

print("\n분석 완료")

Saving data.xlsx to data (52).xlsx

고객 이탈 위험도 분석 결과



,고객 ID,지난달 접속 여부,이번달 접속 여부,지난달 기능 사용,이번달 기능 사용,Churn Score,이탈 위험 단계,분석 평
0,1,1,1,5,6,0.0,충성,활동이 안정적으로 유지되고 있음
1,2,1,1,5,4,12.0,충성,활동이 안정적으로 유지되고 있음
2,3,1,1,4,3,15.0,충성,활동이 안정적으로 유지되고 있음
3,4,1,1,5,2,36.0,일반,일부 활동 감소가 관찰됨
4,5,1,1,4,1,45.0,일반,일부 활동 감소가 관찰됨
5,6,1,1,5,1,48.0,일반,일부 활동 감소가 관찰됨
6,7,1,0,4,0,100.0,이탈,접속 및 기능 사용이 사실상 중단됨
7,8,1,1,4,0,60.0,위험,핵심 기능 사용 감소가 뚜렷함


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


분석 완료
